**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/silkuniene-2025"], check=True)
    os.chdir("/content/encore/entries/silkuniene-2025")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Earlier findings on DANDI:001277, checked at version 0.250701.1610

This notebook checks findings of the earlier attempt on this paper, `silkuniene-2025-03`
(protocol pilot-13, DANDI:001277 at version 0.250701.1610, entry `silkuniene-2025-001277`), that
the blind attempt `silkuniene-2025-04` did not report. It is written by the reconcile step
(docs/protocol.md, "Repeat attempts"), not by the blind attempt's author, and decides no outcome.

Of the earlier attempt's eight findings, six are reported again by `silkuniene-2025-04`
(attempts/silkuniene-2025-04.yaml, `reconciliation`). The two checked here are finding 6 (the
bit depth in the OME metadata) and finding 7 (no electrode positions or ROI coordinates in the
files). Both are checks of metadata and of a central window of each image, streamed from the
archive with the entry's helper module `cancan.py`.

In [ ]:
import re

import h5py
import numpy as np
import pandas as pd

import cancan as cc

DANDISET = "001277"
print("DANDI:%s version %s" % (DANDISET, cc.VERSIONS[DANDISET]))
meta = pd.DataFrame(cc.metadata_table((DANDISET,)))
print(len(meta), "files,", meta["subject"].nunique(), "wells")
print(meta.groupby(["channel", "pulse_ns"]).size().to_string())

## Earlier finding 6: OME bit depth

> The OME metadata in every file give 12 significant bits per pixel, but the YP images hold values
> far above 4095 (the largest 25 x 25 pixel block means are 3500 to 22600), so the stored values
> are not raw 12-bit camera counts. The ImageSeries unit is "n.a." and no scaling is described.

For every file, the OME `SignificantBits` and pixel `Type` (where the file carries OME metadata),
the stored dtype, and the ImageSeries `unit`, `conversion` and `offset` are read. For every
channel of every file, the pixels of a central window of 2000 x 2000 pixels (3.25 mm square,
around the image centre, where the YP spot lies) are read, and their maximum and the fraction
above 4095 are printed.

In [ ]:
HALF = 1000


def bits_and_range(asset):
    f, acq = cc._open(asset["asset_id"])
    d = acq["data"]
    desc = acq.attrs.get("description", "")
    row = dict(subject=asset["subject"], channel=asset["channel"], pulse_ns=asset["pulse_ns"],
               dtype=str(d.dtype),
               significant_bits=";".join(sorted(set(re.findall(r'SignificantBits="(\d+)"', desc)))),
               ome_type=";".join(sorted(set(re.findall(r'Pixels ID="[^"]*"[^>]*Type="(\w+)"', desc)))),
               unit=d.attrs.get("unit"), conversion=float(d.attrs.get("conversion")),
               offset=float(d.attrs.get("offset")))
    r, c = d.shape[-2] // 2, d.shape[-1] // 2
    if d.ndim == 4:
        w = d[0, :, r - HALF:r + HALF, c - HALF:c + HALF]
    else:
        w = d[0, r - HALF:r + HALF, c - HALF:c + HALF][None]
    names = ["Cy3", "DAPI_2h"] if d.ndim == 4 else [asset["channel"]]
    row["windows"] = {n: (int(w[i].max()), float((w[i] > 4095).mean()), int((w[i] == 65535).sum()))
                      for i, n in enumerate(names)}
    f.close()
    return row


bits = pd.DataFrame([bits_and_range(a) for a in meta.to_dict("records")])
print("SignificantBits:", bits["significant_bits"].value_counts().to_dict())
print("OME pixel type:", bits["ome_type"].value_counts().to_dict(), " stored dtype:", bits["dtype"].value_counts().to_dict())
print("ImageSeries unit:", bits["unit"].value_counts().to_dict(),
      " conversion:", bits["conversion"].unique().tolist(), " offset:", bits["offset"].unique().tolist())
print("Files with OME metadata, by channel:",
      bits[bits["significant_bits"] != ""].groupby("channel").size().to_dict(),
      " without:", bits[bits["significant_bits"] == ""].groupby("channel").size().to_dict())
win = pd.DataFrame([dict(subject=b["subject"], pulse_ns=b["pulse_ns"], channel=n, ome_bits=b["significant_bits"],
                         window_max=v[0], frac_above_4095=v[1], pixels_at_65535=v[2])
                    for b in bits.to_dict("records") for n, v in b["windows"].items()])
print()
print("Central 2000 x 2000 pixels, by channel:")
print(win.groupby("channel").agg(files=("subject", "size"), ome_bits=("ome_bits", lambda s: sorted(set(s))),
                                 max_low=("window_max", "min"), max_high=("window_max", "max"),
                                 files_above_4095=("window_max", lambda s: int((s > 4095).sum())),
                                 files_with_65535=("pixels_at_65535", lambda s: int((s > 0).sum()))).to_string())
print()
print("Per file:")
print(win.sort_values(["channel", "pulse_ns", "subject"]).round(4).to_string(index=False))

**Result: still present.** Only the 13 Cy3-DAPI files carry OME metadata, and each gives 12
significant bits and pixel type uint16; the 18 FITC (YP) and 18 DAPI files carry none. In the
central 2000 x 2000 pixels every channel of every file holds values above 4095: YP maxima 16822 to
36988, Cy3 8926 to 25073, the 2-h DAPI channel 19419 to 65535 (pixels at 65535 in 6 of 13) and the
pre-exposure DAPI 40061 to 65535 (in 17 of 18). So the values in the files that state 12 bits are
not 12-bit camera counts. The ImageSeries unit is "n.a." in all 49 files, with conversion 1.0 and
offset 0.0, and no scaling is described. The earlier statement placed the 12 bits in every file;
it is in the Cy3-DAPI files only.

## Earlier finding 7: electrode positions, ROI coordinates and plate layout

> The dandiset gives no electrode positions, ROI coordinates or measured profiles, so the paper's
> ROI placement cannot be repeated exactly; the footprints had to be found in the images. The
> plate layout (wells A1 to B3 on plates P1 to P3) is recoverable only from the subject ids.

Every object stored in every file is listed, and the OME metadata are searched for ROI elements
(`<ROI:` or `<OME:ROI`, beyond the namespace declaration). The text fields (session description,
subject description, ImageSeries description without its OME part, session id) are searched for
words that would give electrode or ROI positions, or the plate and well.

In [ ]:
WORDS = ["coordinate", "position", "roi", "profile", "plate", "well"]


def objects_and_text(asset):
    f, acq = cc._open(asset["asset_id"])
    paths = []
    def walk(group, prefix):
        for k in group:
            if prefix + k == "specifications":
                continue
            paths.append(prefix + k)
            if isinstance(group[k], h5py.Group):
                walk(group[k], prefix + k + "/")
    walk(f, "")
    desc = acq.attrs.get("description", "")
    ome = desc.split("OME metadata:")[-1]
    text = " ".join([f["session_description"][()].decode(),
                     f["general/subject/description"][()].decode(),
                     desc.split("File Metadata")[0],
                     f["general/session_id"][()].decode()]).lower()
    row = dict(subject=asset["subject"], subject_id=f["general/subject/subject_id"][()].decode(),
               paths=paths,
               ome_roi_elements=len(re.findall(r"<(?:ROI|OME):ROI\b", ome)),
               words={w: len(re.findall(r"\b%s" % w, text)) for w in WORDS},
               plate_well_text=re.findall(r"[^.]*\b(?:plate|well)\b[^.]*", text)[:3])
    f.close()
    return row


objs = [objects_and_text(a) for a in meta.to_dict("records")]
all_paths = sorted({p for o in objs for p in o["paths"]})
print("Objects stored in the %d files (specifications left out):" % len(objs))
for p in all_paths:
    print("  ", p, "(in %d files)" % sum(p in o["paths"] for o in objs))
print("Files with OME ROI elements:", sum(o["ome_roi_elements"] > 0 for o in objs))
print("Word counts over all text fields:",
      {w: sum(o["words"][w] for o in objs) for w in WORDS})
print("Sentences naming a plate or well (first file that has one):",
      next((o["plate_well_text"] for o in objs if o["plate_well_text"]), None))
ids = pd.DataFrame([dict(subject_id=o["subject_id"]) for o in objs]).drop_duplicates()
ids["plate"] = ids["subject_id"].str.split("_").str[0]
ids["well"] = ids["subject_id"].str.split("_").str[-1]
print()
print("Plate layout read from the subject ids:")
lay = ids.merge(meta[meta["channel"] == "FITC"][["subject", "pulse_ns", "sham"]].assign(
    subject_id=lambda t: t["subject"].str.replace("-", "_")), on="subject_id", how="left")
print(lay.pivot_table(index="plate", columns="well", values="pulse_ns", aggfunc="first").to_string())

**Result: still present.** Each file holds one image under `acquisition`; `analysis`,
`processing` and `stimulus` are empty, no file has an OME ROI element, and none of the words
coordinate, position, ROI, profile, plate or well appears in the text fields. Electrode positions,
ROI coordinates and profiles are not in the files. The plate (P1 to P3) and well (A1 to B3) are
given only by the subject ids, from which the layout above is read.

## Summary

| Earlier finding | Checked here | Status |
|---|---|---|
| 6, OME bit depth against the stored values | bit depth, unit and central pixel values of all 49 files | still present (12 bits stated in the 13 Cy3-DAPI files only) |
| 7, no electrode positions or ROI coordinates; plate layout only in subject ids | objects and text fields of all 49 files | still present |

The other six earlier findings are reported again by the blind attempt and are not checked here
(attempts/silkuniene-2025-04.yaml, `reconciliation`).